# 0. Environment Setup

## Library Import

Loads all libraries required for the pipeline.

| Library | Purpose |
|---|---|
| `pandas` | DataFrame operations |
| `datetime` | Date math for source folder routing |
| `re` | Regex patterns for column content detection |
| `requests` | Qualtrics API calls |
| `json` | Parse Qualtrics API responses |
| `urllib3` | SSL warning suppression |
| `difflib.get_close_matches` | Fuzzy-match df columns to Qualtrics `questionName` fields |
| `ctypes`, `threading`, `time` | Windows popup notifications (non-blocking) |
| `os` | Build local file paths |

In [2]:
import pandas as pd
from datetime import date, timedelta
import re
import requests
import json
import urllib3
from difflib import get_close_matches
import ctypes
import threading
import time
import os

## Qualtrics Credentials

Sets the API token, data center, and target survey ID.

| Variable | Description |
|---|---|
| `API_TOKEN` | Qualtrics API authentication token |
| `DATA_CENTER` | Qualtrics data center identifier (e.g. `iad1`) |
| `SURVEY_ID` | Qualtrics survey ID for the NY internal survey |

> ⚠️ **Security:** Move credentials to environment variables before sharing or deploying.

In [3]:
# ============================================================
# CONFIGURATION 
# ============================================================

API_TOKEN = "dZZueEgbaPrCuSTXEtIp2sz0EqlJNxg93jYEod7U"   # <-- insert your token
DATA_CENTER = "iad1"
SURVEY_ID = "SV_efytbaOtuAX2uF0"

## Popup Notifications

Defines two Windows message box dialog helpers.

- `popup_info(message)` — success dialog (blue icon)
- `popup_error(message)` — error/warning dialog (red icon)

Each opens in a background thread and auto-closes after `timeout` seconds.

> ⚠️ **Windows only.** Also note: `time.sleep(timeout)` blocks the Jupyter main thread during the timeout. The iQor notebook's version uses `t.join(timeout)` which avoids this.

> ⚠️ Replace with email or Teams notification when deploying to Azure.

In [4]:
def popup_info(message, title="Success", timeout=10):
    MB_OK = 0x0
    MB_ICONINFORMATION = 0x40

    def show_box():
        ctypes.windll.user32.MessageBoxW(0, message, title, MB_OK | MB_ICONINFORMATION)

    t = threading.Thread(target=show_box)
    t.start()

    time.sleep(timeout)

    hwnd = ctypes.windll.user32.FindWindowW(None, title)
    if hwnd:
        ctypes.windll.user32.PostMessageW(hwnd, 0x0010, 0, 0)  # WM_CLOSE


def popup_error(message, title="Error", timeout=10):
    MB_OK = 0x0
    MB_ICONERROR = 0x10

    def show_box():
        ctypes.windll.user32.MessageBoxW(0, message, title, MB_OK | MB_ICONERROR)

    t = threading.Thread(target=show_box)
    t.start()

    time.sleep(timeout)

    hwnd = ctypes.windll.user32.FindWindowW(None, title)
    if hwnd:
        ctypes.windll.user32.PostMessageW(hwnd, 0x0010, 0, 0)


## SharePoint / OneDrive Output Folder

Resolves the local OneDrive sync path where cleaned output files are saved.

- `get_sharepoint_folder()` → full folder path string
- `get_output_path(filename)` → `(full_path, folder_exists_bool)`

Falls back gracefully — if the SharePoint folder is not synced locally the file is saved to the current working directory.

**Hardcoded target folder:**
```
OneDrive - IBERDROLA S.A  General - Customer Research    Post Call Survey      Post Call Survey Data 2025        Avangrid_NY```

In [5]:
def get_onedrive_path():
    return os.path.join(os.path.expanduser("~"), "OneDrive - IBERDROLA S.A")

def get_sharepoint_folder():
    onedrive_root = get_onedrive_path()
    return os.path.join(
        onedrive_root,
        "General - Customer Research",
        "Post Call Survey",
        "Post Call Survey Data 2025",
        "Avangrid_NY"
    )

def get_output_path(filename):
    sharepoint_folder = get_sharepoint_folder()

    if os.path.exists(sharepoint_folder):
        return os.path.join(sharepoint_folder, filename), True

    return filename, False


# 1. Extract

## Extract — Read Excel from Network Share (`extract_data`)

**What it does:** Reads the daily `.xls` file without assuming any fixed column order. Detects each column's semantic meaning by examining the cell content and the question-text row (row 7), then assigns consistent standard column names.

**Input:** `input_path` — UNC path to the `.xls` file on the network share

**Output:** DataFrame with standardized columns

**Source file layout assumed:**
- Row 7 (index 6): Question text labels
- Row 9+ (index 8+): Response data

**Column detection logic:**

| Column | Detection rule |
|---|---|
| `ID` | First value matches `[UE]\d+` |
| `Name` | Column immediately after `ID` |
| `Date/Time` | Column immediately after `Name` |
| `InteractionID` | 12+ alphanumeric chars, not starting with `+` |
| `Phone Number` | Value starts with `+` |
| `Survey Name` | Value contains `"survey"` or `"surv"` |
| `Work Group` | Value contains `"cc"`, `"vendor"`, or `"new"` |
| `NPS` | Question text contains `"recommend"` |
| `FCR` | Question text contains `"resolve"` or `"call back"` |
| `E_H` | Question text contains `"help"` |
| `C_E` | Question text contains `"clear"`, `"explain"`, or `"explaine"` |
| `CSAT` | Question text contains `"satisfied"` |
| `Call Reason` | Question text contains `"payment"`, `"billing"`, or `"outage"` |
| `Unknown_N` | No match — flagged for manual review |

> If a column doesn't match, it's labeled `Unknown_N` rather than crashing — the pipeline continues while flagging the issue.

In [6]:

def extract_data(input_path):

    # Read the full sheet (no assumptions about columns)
    raw = pd.read_excel(input_path, header=None, dtype=str)

    # Row 7 contains the question text
    question_row = raw.iloc[6].fillna("").astype(str)

    # Data starts at row 9
    data = raw.iloc[8:].reset_index(drop=True)

    # Prepare final column names list
    final_cols = []

    for col_idx, col_series in data.items():
        sample_value = col_series.dropna().astype(str).iloc[0] if col_series.dropna().size > 0 else ""
        question_text = question_row[col_idx].lower()

        # -------------------------
        # METADATA COLUMN DETECTION
        # -------------------------

        # ID
        if re.match(r"^[UE]\d+", sample_value):
            final_cols.append("ID")
            continue

        # Name (column immediately right of ID)
        if len(final_cols) > 0 and final_cols[-1] == "ID":
            final_cols.append("Name")
            continue

       # Date/Time (column immediately right of Name)
        if len(final_cols) > 0 and final_cols[-1] == "Name":
            final_cols.append("Date/Time")
            continue

        # InteractionID (alphanumeric)
        if re.match(r"^[A-Za-z0-9]{12,}$", sample_value) and not sample_value.startswith("+"):
            final_cols.append("InteractionID")
            continue

        # Phone Number
        if sample_value.startswith("+"):
            final_cols.append("Phone Number")
            continue

        # Survey Name
        if any(x in sample_value.lower() for x in ["survey", "surv"]):
            final_cols.append("Survey Name")
            continue

        # Work Group
        if any(x in sample_value.lower() for x in ["cc", "vendor", "new"]):
            final_cols.append("Work Group")
            continue

        # -------------------------
        # SCORING COLUMN DETECTION
        # -------------------------

        qt = question_text  # shorthand

        if "recommend" in qt:
            final_cols.append("NPS")
            continue

        if "resolve" in qt or "call back" in qt:
            final_cols.append("FCR")
            continue

        if "help" in qt:
            final_cols.append("E_H")
            continue

        if any(x in qt for x in ["clear", "explain", "explaine"]):
            final_cols.append("C_E")
            continue

        if "satisfied" in qt:
            final_cols.append("CSAT")
            continue

        if any(x in qt for x in ["payment", "billing", "outage"]):
            final_cols.append("Call Reason")
            continue

        # If nothing matches, mark as Unknown
        final_cols.append(f"Unknown_{col_idx}")

    # Apply the detected column names
    data.columns = final_cols

    # Convert Date/Time to proper format
    if "Date/Time" in data.columns:
        data["Date/Time"] = pd.to_datetime(data["Date/Time"], errors="coerce")
        data["Date/Time"] = data["Date/Time"].dt.strftime("%m/%d/%Y %H:%M:%S")
    
    # Convert scoring columns to integers
    score_cols = ["NPS", "FCR", "E_H", "C_E", "CSAT", "Call Reason"]

    for col in score_cols:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors="coerce").astype("Int64")

    
    #display(data.head(5))

    return data



# 2. Transform

## Transform — Validate, Reorder & Compute Status (`transform_data`)

**What it does:**
1. Drops fully-empty columns
2. Warns via popup if any unexpected columns were detected during extract
3. Validates all required score columns are present (raises `ValueError` if missing)
4. Moves `Work Group` to position 3, `CSAT` to position 7
5. Forward-fills `ID` and `Name`
6. Computes `Survey Status`: `"Complete"` if all score columns are filled, `"Abandoned"` otherwise
7. Creates `Tag` column: `"Test"` if `Work Group` contains `"test"` (case-sensitive), else `""`
8. Re-casts score columns to `Int64`

**Input:** Raw DataFrame from `extract_data()`

**Output:** Cleaned, reordered DataFrame with `Survey Status` and `Tag` columns added

**Final column order:**
```
ID | Name | Date/Time | Work Group | InteractionID | Phone Number |
Survey Name | CSAT | NPS | FCR | E_H | C_E | Call Reason | Survey Status | Tag
```

> ⚠️ **Known warning:** Three `SettingWithCopyWarning` messages will appear. Fix by adding `df = df.copy()` at the top of the function.

> ⚠️ **Note:** `Tag` detection here is case-sensitive (`"test"` not `"Test"`). The iQor notebook uses `case=False` — align these to avoid missing test rows.

In [7]:
def transform_data(df):

    # 0. Remove empty columns
    df = df.dropna(axis=1, how="all")

    # 1. Detect unknown columns
    expected_cols = ["ID", "Name", "Date/Time", "InteractionID", "Phone Number",
                     "Survey Name", "Work Group", "NPS", "FCR", "E_H",
                     "C_E", "CSAT", "Call Reason"]

    unknown_cols = [c for c in df.columns if c not in expected_cols]

    # Windows popup warning
    if unknown_cols:
        import ctypes
        message = f"Unknown columns detected:\n{unknown_cols}"
        ctypes.windll.user32.MessageBoxW(0, message, "ETL Warning", 0x40)

    # 2. Validate scoring columns
    required_cols = ["NPS", "FCR", "E_H", "C_E", "CSAT", "Call Reason"]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required scoring columns: {missing}")

    # 3. Validate Work Group and CSAT exist before popping
    if "Work Group" not in df.columns:
        raise ValueError("Column 'Work Group' is missing from the dataset.")

    if "CSAT" not in df.columns:
        raise ValueError("Column 'CSAT' is missing from the dataset.")

    # Move Columns
    csat = df.pop("CSAT")
    work_group = df.pop("Work Group")

    df.insert(3, "Work Group", work_group)
    df.insert(7, "CSAT", csat)

    # Null handling of Name and ID
    df[["ID", "Name"]] = df[["ID", "Name"]].ffill()

    # Create Survey Status/Completion column
    df["Survey Status"] = df[required_cols].notna().all(axis=1)
    df["Survey Status"] = df["Survey Status"].map({True: "Complete", False: "Abandoned"})

    # Tag column
    df["Tag"] = df["Work Group"].str.contains("test", case=True, na=False).map({True: "Test", False: ""})

    # Convert scoring fields into integers
    for col in required_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    #display(df.head(5))
    return df


## Prepare — Qualtrics API Field Mapping (`prepare_for_qualtrics`)

**What it does:**
1. Calls the Qualtrics Survey API to get official question names, question text, and QIDs
2. Fuzzy-matches each DataFrame column to Qualtrics `questionName` values (cutoff: 0.6)
3. Renames columns to their official Qualtrics names
4. Builds the **3-row header block** required by the Qualtrics Import Responses API:
   - Row 1: `questionName`
   - Row 2: `questionText`
   - Row 3: `{"ImportId": "QIDx_TEXT"}`
5. Stacks headers + data into the final upload-ready DataFrame

**Input:** Transformed/tagged DataFrame (all responses — no filter applied here yet)

**Output:** Upload-ready DataFrame — shape `(original_rows + 2, columns)`

> ⚠️ **Missing:** This pipeline does not filter out abandoned/incomplete responses before uploading. See the iQor notebook's `filter_completed()` for the pattern to add here.

In [8]:
def prepare_for_qualtrics(df):

    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    url = f"https://{DATA_CENTER}.qualtrics.com/API/v3/surveys/{SURVEY_ID}"

    headers = {
        "X-API-TOKEN": API_TOKEN
    }

    response = requests.get(url, headers=headers, verify=False)
    
    # Convert API response to JSON
    survey_json = response.json()

    # Pull the Qualtrics question metadata
    label_map = survey_json["result"]["questions"]

    # QID → questionName / questionText
    qid_to_name = {qid: q["questionName"] for qid, q in label_map.items()}
    qid_to_text = {qid: q["questionText"] for qid, q in label_map.items()}

    # questionName → QID
    name_to_qid = {name: qid for qid, name in qid_to_name.items()}

    # All official Qualtrics labels (questionName)
    qualtrics_names = list(name_to_qid.keys())

    # 1) Fuzzy‑match df columns to Qualtrics questionName
    mapped_cols = {}
    used_names = set()

    for col in df.columns:
        match = get_close_matches(col, qualtrics_names, n=1, cutoff=0.6)
        if match:
            new_name = match[0]
            if new_name in used_names:
                new_name = col  # avoid duplicates
            mapped_cols[col] = new_name
            used_names.add(new_name)
        else:
            mapped_cols[col] = col  # leave unmapped as‑is

    df = df.rename(columns=mapped_cols)

    # 2) Row 2: questionText (aligned to final column names)
    row2 = []
    for col in df.columns:
        qid = name_to_qid.get(col)
        row2.append(qid_to_text.get(qid, "") if qid else "")

    # 3) Row 3: {"ImportId": "QIDx_TEXT"} as STRING
    row3 = []
    for col in df.columns:
        qid = name_to_qid.get(col)
        if qid:
            row3.append(f'{{"ImportId": "{qid}_TEXT"}}')
        else:
            row3.append("")

    # 4) Stack as extra rows (no new columns)
    # 4) Stack as extra rows (NO new columns)
    hdr1 = pd.DataFrame([list(df.columns)], columns=df.columns)  # row 1: questionName
    hdr2 = pd.DataFrame([row2], columns=df.columns)              # row 2: questionText
    hdr3 = pd.DataFrame([row3], columns=df.columns)              # row 3: ImportId/QID_TEXT

    df_final = pd.concat([hdr1, hdr2, hdr3, df.reset_index(drop=True)], ignore_index=True)

    # Remove duplicated header row and reset index
    df_final = df_final.iloc[1:].reset_index(drop=True)

   
    display(df_final.head(5))
    return df_final


# 3. Load

## Load — Save DataFrame to CSV (`load_data`)

**What it does:** Writes a DataFrame to a CSV file.

**Input:**
- `df` — any DataFrame
- `output_path` — full file path

**Output:** UTF-8 CSV file written to disk (no index column)

Two files are saved per run:
1. **Repository file** → SharePoint folder (for audit)
2. **Qualtrics temp file** → fixed local path `NY_qualtrics_upload.csv` (uploaded, then can be deleted)

In [9]:
def load_data(df, output_path):
    df.to_csv(output_path, index=False, encoding="utf-8")

    return


## Date Logic — Source Folder Routing

**What it does:** Computes the effective date for building the UNC path to the source `.xls` file. Handles the Monday edge case.

**Input:** System date (`date.today()`)

**Output:** Path component variables

| Variable | Example | Used for |
|---|---|---|
| `source_year` | `2026` | Year folder |
| `source_month` | `"June"` | Month name folder |
| `source_day` | `"15"` | Zero-padded day folder |
| `landing_yesterday` | `2026-06-14` | Output filename suffix |

**Monday handling:** If today is Monday, effective date is set to last Saturday (today − 2 days) since Sunday has no reports.

In [10]:
# Extract Date fields
today = date.today()

# If today is Monday (weekday() == 0), use last Saturday
if today.weekday() == 0:
    effective_date = today - timedelta(days=2)
else:
    effective_date = today

# Extract Date fields
source_year = effective_date.year
source_month = effective_date.strftime("%B")
source_day = effective_date.strftime("%d")
source_weekday = effective_date.weekday()
landing_yesterday = effective_date - timedelta(days=1)

#print(source_year, source_month, source_day, source_weekday, landing_yesterday)


## Upload to Qualtrics (`upload_to_qualtrics`)

**What it does:** POSTs the Qualtrics-ready CSV to the Import Responses API and prints the raw server response.

**Input:**
- `file_path` — path to the Qualtrics-formatted temp CSV
- `DATA_CENTER`, `SURVEY_ID`, `API_TOKEN` — Qualtrics credentials

**Output:** Parsed API response dict (or raw text if JSON parsing fails)

**API endpoint:** `POST /API/v3/surveys/{SURVEY_ID}/import-responses`

> ⚠️ **Missing:** This function returns after the initial `200 - OK` (job accepted), but the import runs asynchronously on Qualtrics's side. Poll `GET .../import-responses/{progressId}` until `status == "complete"` to confirm the upload actually finished. See the iQor notebook's `upload_to_qualtrics` for the polling pattern.

In [11]:
def upload_to_qualtrics(file_path, DATA_CENTER, SURVEY_ID, API_TOKEN):
    """
    Uploads a CSV file to Qualtrics using the Import Responses API.
    Expects a fully formatted Qualtrics-ready CSV at file_path.
    """

    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    url = f"https://{DATA_CENTER}.qualtrics.com/API/v3/surveys/{SURVEY_ID}/import-responses"

    headers = {
        "X-API-TOKEN": API_TOKEN,
        "Content-Type": "text/csv",
        "charset": "UTF-8"
    }

    print(f"\nUploading file to Qualtrics: {file_path}")

    with open(file_path, "rb") as f:
        response = requests.post(
            url,
            headers=headers,
            data=f,
            verify=False
        )

    print("\n=== RAW RESPONSE TEXT ===")
    print(response.text)
    print("=========================\n")

    try:
        result = response.json()
        print("Upload response (parsed):")
        print(json.dumps(result, indent=4))
        return result
    except Exception:
        print("Could not parse JSON response.")
        return response.text


# Execute

Runs the full ETL pipeline end-to-end in a single `try/except` block.

**Flow:**
```
1. Build UNC source path from date variables
2. extract_data(input_path)           → raw DataFrame
3. transform_data(raw_df)             → cleaned + tagged DataFrame
4. load_data(cleaned, sharepoint)     → save repository file
5. prepare_for_qualtrics(cleaned)     → upload-ready DataFrame
6. load_data(ready, temp_path)        → save Qualtrics temp CSV
7. upload_to_qualtrics(temp_path)     → POST to Qualtrics API
```

**Source path pattern:**
```
\\clornas01\DIGITAL_COE_CS_DATA\data_delivery\qualtrics\
  NY_post_call_survey\daily\{year}\{month}\{day}\NY Feedback Daily.xls
```

On success: `popup_info(filename, "Upload successful")`
On failure: `popup_error(error)` with weekday context note if Monday/Sunday

In [12]:
try :
    input_file_path = rf"\\clornas01\DIGITAL_COE_CS_DATA\data_delivery\qualtrics\NY_post_call_survey\daily\{source_year}\{source_month}\{source_day}\NY Feedback Daily.xls"
    #input_file_path = r"\\clornas01\DIGITAL_COE_CS_DATA\data_delivery\qualtrics\NY_post_call_survey\daily\2026\April\03\NY Feedback Daily.xls"
    #output_file_path = r"~\Desktop\NY Feedback Daily Cleaned March 5.csv"
    #output_file_path = "NY_qualtrics_upload6.csv"   # local temp file
    #output_file_path = f"NY Feedback Daily {landing_yesterday}.csv"
    filename = f"NY Feedback Daily {landing_yesterday}.csv"
    output_file_path, used_sharepoint = get_output_path(filename)

    temp_file_path = "NY_qualtrics_upload.csv"   # local temp file
    data = extract_data(input_file_path)
    cleaned_data = transform_data(data)
    load_data(cleaned_data, output_file_path)
    ready_data = prepare_for_qualtrics(cleaned_data)
    load_data(ready_data, temp_file_path)
    result = upload_to_qualtrics(temp_file_path, DATA_CENTER, SURVEY_ID, API_TOKEN)
    
except Exception as e:
    error_message = f"Error running the program:\n{e}"

    if today.weekday() == 0:
        error_message += "\n\nMonday run — Friday folder expected."
    elif today.weekday() == 6:
        error_message += "\n\nWeekend run — folder may be empty."

    popup_error(error_message)

else:
    popup_info(filename,"Upload successful")


C:\Users\E978423\AppData\Local\Temp\ipykernel_18640\848393842.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[["ID", "Name"]] = df[["ID", "Name"]].ffill()
C:\Users\E978423\AppData\Local\Temp\ipykernel_18640\848393842.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Survey Status"] = df[required_cols].notna().all(axis=1)
C:\Users\E978423\AppData\Local\Temp\ipykernel_18640\848393842.py:44: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .lo

,ID,Name,Date/Time,Work Group,InteractionID,Phone Number,Survey Name,CSAT,NPS,FCR,E_H,C_E,Call Reason,Survey Status,Tag
0,ID,Name,Date/Time,Work Group,InteractionID,Phone Number,Survey Name,CSAT,NPS,FCR,Ease of Help,Clear Explanation,Call Reason,Survey Status,Tag
1,"{""ImportId"": ""QID1_TEXT""}","{""ImportId"": ""QID2_TEXT""}","{""ImportId"": ""QID3_TEXT""}","{""ImportId"": ""QID4_TEXT""}","{""ImportId"": ""QID5_TEXT""}","{""ImportId"": ""QID6_TEXT""}","{""ImportId"": ""QID7_TEXT""}","{""ImportId"": ""QID12_TEXT""}","{""ImportId"": ""QID8_TEXT""}","{""ImportId"": ""QID9_TEXT""}","{""ImportId"": ""QID14_TEXT""}","{""ImportId"": ""QID15_TEXT""}","{""ImportId"": ""QID10_TEXT""}","{""ImportId"": ""QID11_TEXT""}","{""ImportId"": ""QID13_TEXT""}"
2,U353049,Sandra Lee,06/29/2026 11:26:58,NYS CC Vendor Transfer Customer Care,200276716570260629,+15854670915,NYS Survey 20260219,<NA>,0,<NA>,<NA>,<NA>,<NA>,Abandoned,
3,U359573,Brookelynn King,06/29/2026 12:08:53,RGE CC New Service Contractor,200276878970260629,+15853183306,RGE Survey 20260219,5,10,1,5,5,6,Complete,
4,U347992,Maurianna Dalton,06/29/2026 12:09:42,NYS CC MIMO,200277198670260629,+16467066927,NYS Survey 20260219,5,4,1,4,5,2,Complete,



Uploading file to Qualtrics: NY_qualtrics_upload.csv

=== RAW RESPONSE TEXT ===
{"result":{"progressId":"c4ddcd24-732f-4502-93fa-fdb1796b06d9","percentComplete":0.0,"status":"inProgress"},"meta":{"requestId":"1ee2eb9e-b3b2-412b-a8d0-39e9d49923d3","httpStatus":"200 - OK"}}

Upload response (parsed):
{
    "result": {
        "progressId": "c4ddcd24-732f-4502-93fa-fdb1796b06d9",
        "percentComplete": 0.0,
        "status": "inProgress"
    },
    "meta": {
        "requestId": "1ee2eb9e-b3b2-412b-a8d0-39e9d49923d3",
        "httpStatus": "200 - OK"
    }
}
